In [0]:

from pyspark.sql import functions as F

catalog = "dbw_healthcare_de_dev"

df_control = spark.table(
    f"{catalog}.etl.file_control"
)

df_audit = spark.table(
    f"{catalog}.etl.file_stage_audit"
)

# Identify files that have not completed processing.
# These are candidates, not yet approved for retry.
df_candidates = (
    df_control
    .filter(
        (F.col("pipeline_status") == "IN_PROGRESS") &
        (F.col("processing_stage").isin("BRONZE", "SILVER", "GOLD")) &
        (F.col("is_enabled") == True)
    )
    .select(
        "file_id",
        "file_name",
        "batch_id",
        "adf_run_id",
        "processing_stage",
        "pipeline_status",
        "retry_count",
        "processing_start_ts"
    )
    .orderBy("processing_start_ts", "file_id")
)

display(df_candidates)


In [0]:

from pyspark.sql import functions as F

recovery_file_ids = [
    "18aa7387-d397-493d-bfb5-d2f8165266f6",
    "e6d607f3-005b-46e9-8aee-0ed32877ba44",
    "8f01bfdd-7ad6-4866-8b5f-1a7df7b6b7db",
    "75ff61d9-9435-4fc7-b944-a4cd68b333cd"
]

df_recovery_audits = (
    spark.table("dbw_healthcare_de_dev.etl.file_stage_audit")
    .filter(F.col("file_id").isin(recovery_file_ids))
    .select(
        "file_id",
        "processing_stage",
        "attempt_number",
        "stage_status",
        "source_record_count",
        "processed_record_count",
        "stage_start_ts",
        "stage_end_ts",
        "error_message"
    )
    .orderBy(
        "file_id",
        "stage_start_ts"
    )
)

display(df_recovery_audits)


In [0]:

from pyspark.sql import functions as F

catalog = "dbw_healthcare_de_dev"

fc = spark.table(f"{catalog}.etl.file_control")
audit = spark.table(f"{catalog}.etl.file_stage_audit")

# Summarize Silver audit history by file.
silver_audit_summary = (
    audit
    .filter(F.col("processing_stage") == "SILVER")
    .groupBy("file_id")
    .agg(
        F.max(
            F.when(F.col("stage_status") == "SUCCESS", 1)
             .otherwise(0)
        ).alias("has_silver_success"),

        F.max(
            F.when(F.col("stage_status") == "IN_PROGRESS", 1)
             .otherwise(0)
        ).alias("has_silver_in_progress"),

        F.max("attempt_number").alias("max_attempt")
    )
)

silver_candidates = (
    fc
    .filter(
        (F.col("processing_stage") == "SILVER") &
        (F.col("pipeline_status") == "IN_PROGRESS") &
        (F.col("is_enabled") == True)
    )
    .join(silver_audit_summary, "file_id", "left")
    .fillna({
        "has_silver_success": 0,
        "has_silver_in_progress": 0,
        "max_attempt": 0
    })
    .withColumn(
        "recovery_classification",
        F.when(
            F.col("has_silver_success") == 1,
            "ALREADY_COMPLETED"
        )
        .when(
            F.col("has_silver_in_progress") == 1,
            "REQUIRES_RUN_STATUS_AND_RECONCILIATION"
        )
        .otherwise("REQUIRES_RECONCILIATION")
    )
    .select(
        "file_name",
        "file_id",
        "adf_run_id",
        "retry_count",
        "max_attempt",
        "has_silver_success",
        "has_silver_in_progress",
        "recovery_classification"
    )
    .orderBy("file_name")
)

display(silver_candidates)


In [0]:

from pyspark.sql import functions as F

catalog = "dbw_healthcare_de_dev"

bronze = spark.table(
    f"{catalog}.bronze.patients_raw"
)

silver = spark.table(
    f"{catalog}.silver.DimPatients"
)

recovery_file_ids = [
    "e6d607f3-005b-46e9-8aee-0ed32877ba44",  # Batch 5
    "75ff61d9-9435-4fc7-b944-a4cd68b333cd"   # Batch 7
]

# Get the patient IDs associated with each source file.
source_patients = (
    bronze
    .filter(F.col("file_id").isin(recovery_file_ids))
    .select(
        "file_id",
        F.col("batch_id").alias("source_batch_id"),
        F.col("patient_id").alias("PatientId")
    )
    .distinct()
)

# Match source patients to existing Silver versions.
reconciliation = (
    source_patients.alias("b")
    .join(
        silver.alias("s"),
        F.col("b.PatientId") == F.col("s.PatientId"),
        "left"
    )
    .select(
        F.col("b.file_id"),
        F.col("b.source_batch_id"),
        F.col("b.PatientId"),
        F.col("s.BatchId").alias("silver_batch_id"),
        F.col("s.State").alias("silver_state"),
        F.col("s.LastName").alias("silver_last_name"),
        F.col("s.IsActive"),
        F.col("s.RowStartDate")
    )
    .orderBy(
        "file_id",
        "PatientId",
        "RowStartDate"
    )
)

display(reconciliation)


In [0]:

from pyspark.sql import functions as F

catalog = "dbw_healthcare_de_dev"

batch5_file_id = "e6d607f3-005b-46e9-8aee-0ed32877ba44"

bronze_batch5 = (
    spark.table(f"{catalog}.bronze.patients_raw")
    .filter(F.col("file_id") == batch5_file_id)
    .alias("b")
)

silver_batch5 = (
    spark.table(f"{catalog}.silver.DimPatients")
    .alias("s")
)

comparison = (
    bronze_batch5
    .join(
        silver_batch5,
        (F.col("b.patient_id") == F.col("s.PatientId")) &
        (F.col("b.batch_id") == F.col("s.BatchId")),
        "left"
    )
    .select(
        F.col("b.patient_id").alias("PatientId"),
        F.col("b.batch_id").alias("BatchId"),
        F.col("b.first_name").alias("BronzeFirstName"),
        F.col("s.FirstName").alias("SilverFirstName"),
        F.col("b.last_name").alias("BronzeLastName"),
        F.col("s.LastName").alias("SilverLastName"),
        F.col("b.gender").alias("BronzeGender"),
        F.col("s.Gender").alias("SilverGender"),
        F.col("b.date_of_birth").alias("BronzeDOB"),
        F.col("s.DateOfBirth").alias("SilverDOB"),
        F.col("b.state").alias("BronzeState"),
        F.col("s.State").alias("SilverState"),
        F.col("s.IsActive")
    )
    .withColumn(
        "values_match",
        F.col("BronzeFirstName").eqNullSafe(F.col("SilverFirstName")) &
        F.col("BronzeLastName").eqNullSafe(F.col("SilverLastName")) &
        F.col("BronzeGender").eqNullSafe(F.col("SilverGender")) &
        F.col("BronzeDOB").eqNullSafe(F.col("SilverDOB")) &
        F.col("BronzeState").eqNullSafe(F.col("SilverState"))
    )
)

display(comparison)
